# 预训练 HyenaDNA d256＋变异中心表示：真实监督与 Evo 2 蒸馏实验

使用官方固定版本 `LongSafari/hyenadna-tiny-1k-seqlen-d256-hf`，共享 REF/ALT encoder，
提取变异索引 128 的最后隐藏层，拼接 `ref, alt, alt-ref`（768 维）。
完整推理模型 1,854,465 参数，编码器与分类头均微调；因果中心表示只使用当前位置及左侧上下文。

预先固定本轮配置 `configs/hyenadna_d256.yaml`，相同真实预训练权重、随机分类头初值和数据划分，
对比仅监督与监督＋特征/预测蒸馏。复用既有真实 NVIDIA Evo 2 **40B** 特征和已训练 Teacher；
此 Notebook 不调用 NVIDIA API、不重训 Teacher、不读取 API key、不使用合成标签报告性能。

数据为现有 2,000 条 ClinVar GRCh38 常染色体 SNV；训练/验证/测试为 1,200/400/400。
只按验证 BCE 早停，测试集不参与训练或选超参数。该测试集已在历史 CNN 实验中查看过，
本轮结果属于同一研究队列的后续比较，不能充当全新外部验证。

在项目根目录 PowerShell 首次安装并执行：
```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-hyenadna.txt -r requirements-notebook.txt
.\.venv\Scripts\python.exe scripts/run_hyenadna_notebook.py
```
执行后的含输出版本保存为项目根目录 `HyenaDNA_d256_experiment.executed.ipynb`。
如需交互查看：`.\.venv\Scripts\python.exe -m notebook HyenaDNA_d256_experiment.executed.ipynb`。
所有 cell 的 CLI 均由当前项目 .venv 启动，stdout 同步写入阶段日志。
发生错误时保存部分 Notebook 并抛出异常；修复后重跑将严格校验并复用已完成阶段。
中断中的训练阶段会从固定预训练初始化重新开始，未实现 epoch 中间优化器状态恢复。

In [1]:
import os, sys, json, csv
from pathlib import Path
from datetime import datetime, timezone
ROOT = Path.cwd().resolve()
assert (ROOT / "src/evo2_mvp").is_dir(), "请从项目根目录执行"
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve(), sys.executable
sys.path.insert(0, str(ROOT / "scripts"))
os.environ["HF_HOME"] = str(ROOT / "work/huggingface")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch, transformers, safetensors, numpy as np
from hyenadna_notebook_support import input_identity, run_stage, validate_training
from evo2_mvp.utils import save_json, sha256_file
from evo2_mvp.train_teacher_head import experiment_data, check_cache
from evo2_mvp.cache import FeatureCache
from evo2_mvp.distill_student import check_phase0
RUN_ROOT = Path("outputs/hyenadna_notebook_run_20261104")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SUPERVISED = RUN_ROOT / "supervised"
DISTILLED = RUN_ROOT / "distilled"
torch.set_num_threads(4)
print(json.dumps({"python": sys.version, "executable": sys.executable, "torch": torch.__version__,
                  "transformers": transformers.__version__, "device": "cpu", "run_root": str(RUN_ROOT)}, indent=2))

D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "python": "3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]",
  "executable": "D:\\Users\\60152\\learning materials\\CISC7021\\Report\\evo2_pathogenicity_mvp\\.venv\\Scripts\\python.exe",
  "torch": "2.7.1+cpu",
  "transformers": "4.46.3",
  "device": "cpu",
  "run_root": "outputs\\hyenadna_notebook_run_20261104"
}


In [2]:
variants, splits, provenance = experiment_data("data/variants.csv", "data/splits.json", False)
cache = FeatureCache("outputs/nvidia_cache")
check_cache(cache, provenance)
check_phase0("outputs/phase0_api_verified/result.json", cache.metadata)
# This hashes the actual ~3.3GB FASTA, not merely its filename.
IDENTITY = input_identity(ROOT)
assert cache.metadata["reference_sha256"] == IDENTITY["files_sha256"]["data/GRCh38.fa"]
assert len(variants) == 2000 and provenance["synthetic"] is False
dataset = {"provenance": provenance, "splits": {s: {"n": len(splits[s]),
    "positives": sum(v.label for v in variants if v.id in set(splits[s]))} for s in ("train", "val", "test")},
    "experiment_identity": IDENTITY["identity"], "new_api_calls_required": 0}
save_json(RUN_ROOT / "input_identity.json", IDENTITY)
save_json(RUN_ROOT / "data_check.json", dataset)
print(json.dumps(dataset, indent=2))

{
  "provenance": {
    "variants_sha256": "b49c81a5d5e18f4230d34f51870a44fab514e976c54635388c4bf3b8abd8f848",
    "variant_label_fingerprint": "1138fbfbe6f18b6de0b46b877e9b9cf0e7c40bcb76f8248b1987bbcb2d55051a",
    "n_variants": 2000,
    "synthetic": false,
    "sources": [
      "ClinVar"
    ],
    "performance_scope": "research_dataset"
  },
  "splits": {
    "train": {
      "n": 1200,
      "positives": 183
    },
    "val": {
      "n": 400,
      "positives": 74
    },
    "test": {
      "n": 400,
      "positives": 48
    }
  },
  "experiment_identity": "96ebfc7fad4467fd12eb75f2facba8b01dac254e522f6f0094a241ace856c72d",
  "new_api_calls_required": 0
}


## 固定真实预训练权重与前向验证

以下脚本只接受固定 revision 和 SHA256；已有正确文件时离线复核，不重新下载。
结构测试使用固定合成 DNA，仅检查真实 embedding、梯度与保存重载，不报告随机分类头分数。
```powershell
.\.venv\Scripts\python.exe scripts/prepare_hyenadna.py --output work/hyenadna_d256_pretrained
.\.venv\Scripts\python.exe scripts/test_hyenadna.py --pretrained-dir work/hyenadna_d256_pretrained --window 256 --threads 4 --compare-official --output outputs/hyenadna_notebook_run_20261104/phase0
```

In [3]:
from evo2_mvp.hyenadna import PINNED_FILES
run_stage(ROOT, RUN_ROOT, "prepare", ["scripts/prepare_hyenadna.py", "--output", "work/hyenadna_d256_pretrained"],
          IDENTITY, [str(Path("work/hyenadna_d256_pretrained") / name) for name in PINNED_FILES])

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u scripts/prepare_hyenadna.py --output work/hyenadna_d256_pretrained


model.safetensors: existing_sha256_verified


config.json: existing_sha256_verified


configuration_hyena.py: existing_sha256_verified


modeling_hyena.py: existing_sha256_verified


tokenization_hyena.py: existing_sha256_verified


... (6 outputs left out)

  "directory": "D:\\Users\\60152\\learning materials\\CISC7021\\Report\\evo2_pathogenicity_mvp\\work\\hyenadna_d256_pretrained",


  "all_sha256_verified": true


}


{'status': 'completed',
 'signature': {'experiment_identity': '96ebfc7fad4467fd12eb75f2facba8b01dac254e522f6f0094a241ace856c72d',
  'arguments': ['scripts/prepare_hyenadna.py',
   '--output',
   'work/hyenadna_d256_pretrained']},
 'started_at_utc': '2026-11-03T16:49:51.089528+00:00',
 'completed_at_utc': '2026-11-03T16:49:51.695406+00:00',
 'returncode': 0,
 'artifacts_sha256': {'work/hyenadna_d256_pretrained/model.safetensors': '81c15c4042d5fae646f00b429b76c65644cdbe88aa7e98b61aa781c84ff09031',
  'work/hyenadna_d256_pretrained/config.json': '2b3209dc6d176d0cf573e0cdadbedb4ab0c5f7456136474200c5d827a126d336',
  'work/hyenadna_d256_pretrained/configuration_hyena.py': '29cb6cae67f5010e279d67349e688663043f0bd6791a09c36d96da8cac3668a6',
  'work/hyenadna_d256_pretrained/modeling_hyena.py': 'd78029cacea2c9259f1293fdb50b60f90872089469185e27d5c769086c475607',
  'work/hyenadna_d256_pretrained/tokenization_hyena.py': 'd424da2f794958b0360e033bbf5edc9ee4e3e3126e883093621348f07761e408',
  'work/hyen

In [4]:
def validate_smoke():
    report = json.loads((RUN_ROOT / "phase0/result.json").read_text(encoding="utf-8"))
    assert report["status"] == "passed" and report["pretrained_encoder_loaded"] is True
    assert report["variant_features_shape"] == [768] and report["finite"] is True
    assert report["official_implementation_comparison"]["status"] == "passed"
    assert report["save_reload_features_bitwise_equal"] and report["encoder_backward_finite_nonzero"]
run_stage(ROOT, RUN_ROOT, "phase0", ["scripts/test_hyenadna.py", "--pretrained-dir", "work/hyenadna_d256_pretrained",
    "--window", "256", "--threads", "4", "--compare-official", "--output", str(RUN_ROOT / "phase0")], IDENTITY,
    [str(RUN_ROOT / "phase0" / name) for name in ["result.json", "embedding_ref.npy", "embedding_alt.npy",
    "embedding_delta.npy", "variant_features.npy"]], validate_smoke)

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u scripts/test_hyenadna.py --pretrained-dir work/hyenadna_d256_pretrained --window 256 --threads 4 --compare-official --output outputs\hyenadna_notebook_run_20261104\phase0


{


  "status": "passed",


  "repo_id": "LongSafari/hyenadna-tiny-1k-seqlen-d256-hf",


  "revision": "6036d4e144922b44470294f510cbfd2539ba3b7e",


  "parameter_count": 1854465,


... (18 outputs left out)

  }


}


Saved D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\outputs\hyenadna_notebook_run_20261104\phase0\result.json


{'status': 'completed',
 'signature': {'experiment_identity': '96ebfc7fad4467fd12eb75f2facba8b01dac254e522f6f0094a241ace856c72d',
  'arguments': ['scripts/test_hyenadna.py',
   '--pretrained-dir',
   'work/hyenadna_d256_pretrained',
   '--window',
   '256',
   '--threads',
   '4',
   '--compare-official',
   '--output',
   'outputs\\hyenadna_notebook_run_20261104\\phase0']},
 'started_at_utc': '2026-11-03T16:49:51.746172+00:00',
 'completed_at_utc': '2026-11-03T16:50:00.134073+00:00',
 'returncode': 0,
 'artifacts_sha256': {'outputs/hyenadna_notebook_run_20261104/phase0/result.json': 'cf42d48a65baae3e0e86d70a2c05372365fbcd9ea12ad40763c1101e2412a5f9',
  'outputs/hyenadna_notebook_run_20261104/phase0/embedding_ref.npy': '6a2faf8519a3d4a27b4abc5dd5112c2e8aaf88f02e7f0fbc6a2c2782bbe27866',
  'outputs/hyenadna_notebook_run_20261104/phase0/embedding_alt.npy': '0897e4bafba86337f0d3a396bd53edd96d1221fb5bf3316506a99253ada090ee',
  'outputs/hyenadna_notebook_run_20261104/phase0/embedding_delta.np

In [5]:
os.environ["RUN_HYENADNA_INTEGRATION"] = "1"
# Real local checkpoint integration is enabled; a fresh NVIDIA API integration is deliberately not requested.
run_stage(ROOT, RUN_ROOT, "tests", ["-m", "pytest", "-q", "--basetemp", "work/pytest_notebook_run",
    "--junitxml", str(RUN_ROOT / "junit.xml")], IDENTITY, [str(RUN_ROOT / "junit.xml")])

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m pytest -q --basetemp work/pytest_notebook_run --junitxml outputs\hyenadna_notebook_run_20261104\junit.xml


..........s............................................................. [ 50%]


........................................................................ [100%]


- generated xml file: D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\outputs\hyenadna_notebook_run_20261104\junit.xml -


143 passed, 1 skipped in 71.35s (0:01:11)


{'status': 'completed',
 'signature': {'experiment_identity': '96ebfc7fad4467fd12eb75f2facba8b01dac254e522f6f0094a241ace856c72d',
  'arguments': ['-m',
   'pytest',
   '-q',
   '--basetemp',
   'work/pytest_notebook_run',
   '--junitxml',
   'outputs\\hyenadna_notebook_run_20261104\\junit.xml']},
 'started_at_utc': '2026-11-03T16:50:00.184632+00:00',
 'completed_at_utc': '2026-11-03T16:51:13.986433+00:00',
 'returncode': 0,
 'artifacts_sha256': {'outputs/hyenadna_notebook_run_20261104/junit.xml': 'ef7a88875e81bd6599e74cb09d2dc99d071fd36b44e8105dab7bbfaae9cd3f13'}}

## 两个同初始化训练实验

编码器学习率 2e-5，分类头 1e-3，batch 16，最多 20 epochs，patience 5，梯度裁剪 1.0。
蒸馏初值 alpha/beta/gamma=1.0/0.1/0.5，T=2；Teacher 投影仅用训练统计并固定。
两个实验沿用同一配置，全部参数及随机初值 SHA 写入 checkpoint，比较阶段强制核对。
每轮 `student_epoch` JSON 包含训练损失、验证 BCE、梯度范数和耗时。
日志 `outputs/hyenadna_notebook_run_20261104/supervised.log` 与 `distilled.log` 可实时查看。

In [6]:
COMMON = ["--variants", "data/variants.csv", "--splits", "data/splits.json", "--cache", "outputs/nvidia_cache",
    "--fasta", "data/GRCh38.fa", "--teacher", "outputs/teacher/best_teacher.pt",
    "--phase0-report", "outputs/phase0_api_verified/result.json", "--config", "configs/hyenadna_d256.yaml", "--device", "cpu"]
artifacts = [str(SUPERVISED / name) for name in ["best_student.pt", "metrics.json", "predictions.csv",
    "prediction_metadata.json", "benchmark.json", "training_curve.csv", "training_curve.png", "config.json"]]
run_stage(ROOT, RUN_ROOT, "supervised", ["-m", "evo2_mvp.distill_student", "--mode", "supervised",
    *COMMON, "--output", str(SUPERVISED)], IDENTITY, artifacts,
    lambda: validate_training(SUPERVISED, "supervised", IDENTITY))

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m evo2_mvp.distill_student --mode supervised --variants data/variants.csv --splits data/splits.json --cache outputs/nvidia_cache --fasta data/GRCh38.fa --teacher outputs/teacher/best_teacher.pt --phase0-report outputs/phase0_api_verified/result.json --config configs/hyenadna_d256.yaml --device cpu --output outputs\hyenadna_notebook_run_20261104\supervised


{"event": "student_epoch", "mode": "supervised", "epoch": 1, "train_loss": 0.44767736275990805, "val_loss": 0.4761832356452942, "train_supervised": 0.44767736275990805, "train_feature": 0.0, "train_prediction_kd": 0.0, "gradient_norm_before_clip": 1.7658249004681905, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 52.540837299995474}


{"event": "student_epoch", "mode": "supervised", "epoch": 2, "train_loss": 0.4160093809167544, "val_loss": 0.4920766055583954, "train_supervised": 0.4160093809167544, "train_feature": 0.0, "train_prediction_kd": 0.0, "gradient_norm_before_clip": 4.5106350255012515, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 51.76739569999336}


{"event": "student_epoch", "mode": "supervised", "epoch": 3, "train_loss": 0.3135012904802958, "val_loss": 0.5598188638687134, "train_supervised": 0.3135012904802958, "train_feature": 0.0, "train_prediction_kd": 0.0, "gradient_norm_before_clip": 11.114123090108235, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 51.48445949998859}


{"event": "student_epoch", "mode": "supervised", "epoch": 4, "train_loss": 0.15325991176689666, "val_loss": 0.8153259754180908, "train_supervised": 0.15325991176689666, "train_feature": 0.0, "train_prediction_kd": 0.0, "gradient_norm_before_clip": 21.25401008605957, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 52.17142349999631}


{"event": "student_epoch", "mode": "supervised", "epoch": 5, "train_loss": 0.07314803170583521, "val_loss": 1.1713051795959473, "train_supervised": 0.07314803170583521, "train_feature": 0.0, "train_prediction_kd": 0.0, "gradient_norm_before_clip": 16.852270185450713, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 51.055589199997485}


... (30 outputs left out)

  },


  "performance_scope": "research_dataset"


}


{'status': 'completed',
 'signature': {'experiment_identity': '96ebfc7fad4467fd12eb75f2facba8b01dac254e522f6f0094a241ace856c72d',
  'arguments': ['-m',
   'evo2_mvp.distill_student',
   '--mode',
   'supervised',
   '--variants',
   'data/variants.csv',
   '--splits',
   'data/splits.json',
   '--cache',
   'outputs/nvidia_cache',
   '--fasta',
   'data/GRCh38.fa',
   '--teacher',
   'outputs/teacher/best_teacher.pt',
   '--phase0-report',
   'outputs/phase0_api_verified/result.json',
   '--config',
   'configs/hyenadna_d256.yaml',
   '--device',
   'cpu',
   '--output',
   'outputs\\hyenadna_notebook_run_20261104\\supervised']},
 'started_at_utc': '2026-11-03T16:51:14.048943+00:00',
 'completed_at_utc': '2026-11-03T16:57:37.296152+00:00',
 'returncode': 0,
 'artifacts_sha256': {'outputs/hyenadna_notebook_run_20261104/supervised/best_student.pt': '7db00d61ebe68d0ac3ca600788fe84b6b3e37c47633cf396c5bfeb4c4f207dd6',
  'outputs/hyenadna_notebook_run_20261104/supervised/metrics.json': '9a16

In [7]:
artifacts = [str(DISTILLED / name) for name in ["best_student.pt", "metrics.json", "predictions.csv",
    "prediction_metadata.json", "benchmark.json", "training_curve.csv", "training_curve.png", "config.json",
    "distillation_training_state.pt"]]
run_stage(ROOT, RUN_ROOT, "distilled", ["-m", "evo2_mvp.distill_student", "--mode", "distill",
    *COMMON, "--output", str(DISTILLED)], IDENTITY, artifacts,
    lambda: validate_training(DISTILLED, "distill", IDENTITY))

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m evo2_mvp.distill_student --mode distill --variants data/variants.csv --splits data/splits.json --cache outputs/nvidia_cache --fasta data/GRCh38.fa --teacher outputs/teacher/best_teacher.pt --phase0-report outputs/phase0_api_verified/result.json --config configs/hyenadna_d256.yaml --device cpu --output outputs\hyenadna_notebook_run_20261104\distilled


{"event": "student_epoch", "mode": "distill", "epoch": 1, "train_loss": 1.5010485601425172, "val_loss": 0.4781181216239929, "train_supervised": 0.47638428181409836, "train_feature": 0.013362588956952096, "train_prediction_kd": 2.0466560252507526, "gradient_norm_before_clip": 4.359549418290456, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 53.77595460000157}


{"event": "student_epoch", "mode": "distill", "epoch": 2, "train_loss": 1.423419202963511, "val_loss": 0.5373443365097046, "train_supervised": 0.44519599080085753, "train_feature": 0.011211775466799736, "train_prediction_kd": 1.954204068183899, "gradient_norm_before_clip": 9.849149815241496, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 51.92538990000321}


{"event": "student_epoch", "mode": "distill", "epoch": 3, "train_loss": 1.218193649450938, "val_loss": 0.5887430310249329, "train_supervised": 0.34893381436665855, "train_feature": 0.010383839023609957, "train_prediction_kd": 1.736442918777466, "gradient_norm_before_clip": 22.40135603586833, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 57.15299679999589}


{"event": "student_epoch", "mode": "distill", "epoch": 4, "train_loss": 0.8841875779628754, "val_loss": 0.7531572580337524, "train_supervised": 0.18836677499115467, "train_feature": 0.009836997011055549, "train_prediction_kd": 1.3896741970380146, "gradient_norm_before_clip": 37.052459716796875, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 55.6233315999998}


{"event": "student_epoch", "mode": "distill", "epoch": 5, "train_loss": 0.6949325851599375, "val_loss": 0.7809080481529236, "train_supervised": 0.09993071588377157, "train_feature": 0.009468608361979325, "train_prediction_kd": 1.1881100225448609, "gradient_norm_before_clip": 36.23442081451416, "encoder_lr": 2e-05, "head_lr": 0.001, "epoch_seconds": 55.2425002999953}


... (30 outputs left out)

  },


  "performance_scope": "research_dataset"


}


{'status': 'completed',
 'signature': {'experiment_identity': '96ebfc7fad4467fd12eb75f2facba8b01dac254e522f6f0094a241ace856c72d',
  'arguments': ['-m',
   'evo2_mvp.distill_student',
   '--mode',
   'distill',
   '--variants',
   'data/variants.csv',
   '--splits',
   'data/splits.json',
   '--cache',
   'outputs/nvidia_cache',
   '--fasta',
   'data/GRCh38.fa',
   '--teacher',
   'outputs/teacher/best_teacher.pt',
   '--phase0-report',
   'outputs/phase0_api_verified/result.json',
   '--config',
   'configs/hyenadna_d256.yaml',
   '--device',
   'cpu',
   '--output',
   'outputs\\hyenadna_notebook_run_20261104\\distilled']},
 'started_at_utc': '2026-11-03T16:57:37.338208+00:00',
 'completed_at_utc': '2026-11-03T17:04:15.624264+00:00',
 'returncode': 0,
 'artifacts_sha256': {'outputs/hyenadna_notebook_run_20261104/distilled/best_student.pt': '1e7595afa1817a8c921f2740583a70eb59bb9865695315e98fc03ee68b0d6fdc',
  'outputs/hyenadna_notebook_run_20261104/distilled/metrics.json': '01e89717b5

In [8]:
run_stage(ROOT, RUN_ROOT, "comparison", ["scripts/compare_models.py", "--teacher", "outputs/teacher",
    "--supervised", str(SUPERVISED), "--distilled", str(DISTILLED), "--output", str(RUN_ROOT / "comparison.json")],
    IDENTITY, [str(RUN_ROOT / "comparison.json")])
comparison = json.loads((RUN_ROOT / "comparison.json").read_text(encoding="utf-8"))
assert comparison["controlled_distillation_comparison"] is True
assert comparison["same_independent_test_set"] is True
print("注意：缓存 Teacher 分类头耗时不包含远程 Evo 2，不能拿它计算 Student 整体加速倍数。")

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u scripts/compare_models.py --teacher outputs/teacher --supervised outputs\hyenadna_notebook_run_20261104\supervised --distilled outputs\hyenadna_notebook_run_20261104\distilled --output outputs\hyenadna_notebook_run_20261104\comparison.json


{


  "models": {


    "teacher": {


      "test": {


        "n": 400,


... (199 outputs left out)

  "latency_note": "Cached teacher head latency excludes Evo 2; use full warm prediction benchmarks for comparable inference cost.",


  "clinical_use": "research_only"


}


注意：缓存 Teacher 分类头耗时不包含远程 Evo 2，不能拿它计算 Student 整体加速倍数。

## 独立学生推理与重载检查

使用完整训练后 checkpoint，不使用原始预训练目录、Teacher 或 API。
这里再次移除子进程 NVIDIA key 环境变量以验证独立运行；不修改用户 .env。
```powershell
.\.venv\Scripts\python.exe -m evo2_mvp.inference --fasta data/GRCh38.fa --chrom 1 --pos 926029 --ref C --alt T --student outputs/hyenadna_notebook_run_20261104/distilled/best_student.pt --device cpu --cpu-threads 4 --output outputs/hyenadna_notebook_run_20261104/prediction.json
```
新模型尚未校准，不能复用旧 CNN 温度。熵单位为 nats，不等于临床置信度。

In [9]:
for name in ("NVIDIA_API_KEY", "NVIDIA_APIKEY", "NVAPI_KEY"):
    os.environ.pop(name, None)
for suffix in ("prediction", "prediction_reloaded"):
    run_stage(ROOT, RUN_ROOT, suffix, ["-m", "evo2_mvp.inference", "--fasta", "data/GRCh38.fa",
        "--chrom", "1", "--pos", "926029", "--ref", "C", "--alt", "T", "--student", str(DISTILLED / "best_student.pt"),
        "--device", "cpu", "--cpu-threads", "4", "--output", str(RUN_ROOT / f"{suffix}.json")],
        IDENTITY, [str(RUN_ROOT / f"{suffix}.json")])
prediction = json.loads((RUN_ROOT / "prediction.json").read_text(encoding="utf-8"))
assert prediction == json.loads((RUN_ROOT / "prediction_reloaded.json").read_text(encoding="utf-8"))
assert 0 <= prediction["student_score"] <= 1 and np.isfinite(prediction["student_predictive_entropy"])
assert prediction["teacher_score"] is None and prediction["clinical_use"] == "research_only"
print(json.dumps(prediction, indent=2))

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m evo2_mvp.inference --fasta data/GRCh38.fa --chrom 1 --pos 926029 --ref C --alt T --student outputs\hyenadna_notebook_run_20261104\distilled\best_student.pt --device cpu --cpu-threads 4 --output outputs\hyenadna_notebook_run_20261104\prediction.json


{


  "variant_id": "1-926029-C-T",


  "genome_build": "GRCh38",


  "teacher_score": null,


  "student_score": 0.11535679548978806,


... (15 outputs left out)

  "calibration_status": "not_calibrated",


  "clinical_use": "research_only"


}


{
  "variant_id": "1-926029-C-T",
  "genome_build": "GRCh38",
  "teacher_score": null,
  "student_score": 0.11535679548978806,
  "student_predictive_entropy": 0.35757049022074805,
  "ensemble_std": null,
  "calibrated_score": null,
  "calibration_status": "not_calibrated",
  "clinical_use": "research_only"
}


In [10]:
import pandas as pd
from IPython.display import display
from evo2_mvp.models import load_checkpoint
records = []
for name, directory in [("Evo 2 + MLP", Path("outputs/teacher")),
                        ("历史 CNN 监督", Path("outputs/student_supervised")),
                        ("历史 CNN 蒸馏", Path("outputs/student_distilled")),
                        ("HyenaDNA 监督", SUPERVISED), ("HyenaDNA 蒸馏", DISTILLED)]:
    m = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))["test"]
    records.append({"model": name, **{k: m[k] for k in ("auroc", "auprc", "f1", "recall", "brier_score", "n", "positives")}})
table = pd.DataFrame(records)
table.to_csv(RUN_ROOT / "model_comparison.csv", index=False)
display(table)
epochs = {}
for name, directory in [("supervised", SUPERVISED), ("distilled", DISTILLED)]:
    c = load_checkpoint(directory / "best_student.pt")
    epochs[name] = {"epochs_run": c["n_epochs_run"], "best_epoch": c["best_epoch"],
                    "checkpoint_sha256": sha256_file(directory / "best_student.pt")}
report = {"status": "completed", "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_identity": IDENTITY["identity"], "data": dataset, "epochs": epochs,
    "student_parameter_count": 1854465, "embedding_dim": 256, "fused_feature_dim": 768,
    "test_comparison": records, "same_initialization_verified": True, "standalone_prediction": prediction,
    "standalone_reload_prediction_identical": True, "new_nvidia_api_calls": 0,
    "limitations": ["single seed", "only 1200 labeled training variants", "previously examined internal test cohort",
                    "causal center does not see downstream context", "research_only; not clinical validation"],
    "clinical_use": "research_only"}
save_json(RUN_ROOT / "run_report.json", report)
print("实验已真实完成；性能以表中结果为准，不保证蒸馏改善。报告:", RUN_ROOT / "run_report.json")

,model,auroc,auprc,f1,recall,brier_score,n,positives
0,Evo 2 + MLP,0.937086,0.779759,0.680412,0.6875,0.063894,400,48
1,历史 CNN 监督,0.561494,0.197017,0.000000,0.0000,0.107098,400,48
2,历史 CNN 蒸馏,0.560606,0.173807,0.000000,0.0000,0.107440,400,48
3,HyenaDNA 监督,0.567649,0.171754,0.000000,0.0000,0.108085,400,48
4,HyenaDNA 蒸馏,0.581499,0.168386,0.000000,0.0000,0.105020,400,48


实验已真实完成；性能以表中结果为准，不保证蒸馏改善。报告: outputs\hyenadna_notebook_run_20261104\run_report.json
